In [1]:
import psycopg2
from psycopg2 import sql

def connect_to_db():
    try:
        conn = psycopg2.connect(
            dbname="GPUPhotDB",
            user="read_only",
            password="read_only",
            host="postgres",
            port="5432"
        )
        return conn
    except (Exception, psycopg2.Error) as error:
        print("Error conectando a PostgreSQL:", error)
        return None

In [2]:
def get_tables_and_columns():
    conn = connect_to_db()
    if conn is None:
        return None
    
    try:
        cur = conn.cursor()
        
        # Consulta para obtener todas las tablas y sus columnas
        query = """
            SELECT table_name, column_name
            FROM information_schema.columns
            WHERE table_schema = 'public'
            ORDER BY table_name, ordinal_position;
        """
        
        cur.execute(query)
        results = cur.fetchall()
        
        # Organizar los resultados en un diccionario
        tables_dict = {}
        for table_name, column_name in results:
            if table_name not in tables_dict:
                tables_dict[table_name] = []
            tables_dict[table_name].append(column_name)
        
        return tables_dict
    except (Exception, psycopg2.Error) as error:
        print("Error ejecutando la consulta:", error)
        return {}
    finally:
        if conn:
            cur.close()
            conn.close()

tables_and_columns = get_tables_and_columns()
for table, columns in tables_and_columns.items():
    print(f"Tabla: {table}, Columnas: {columns}")


Tabla: imaphot, Columnas: ['id', 'ra', 'dec', 'flux', 'dflux', 'trans']
Tabla: imastats, Columnas: ['id', 'file_path', 'naxis1', 'naxis2', 'telescop', 'instrume', 'camera', 'filter', 'date_obs', 'exptime', 'object', 'ra', 'dec', 'fwhm', 'maglim', 'header']


In [3]:
import pandas as pd

def results_to_dataframe(results, column_names):
    """
    Convierte los resultados de una consulta en un DataFrame de pandas.
    
    :param results: Lista de tuplas con los resultados de la consulta
    :param column_names: Lista con los nombres de las columnas
    :return: DataFrame de pandas
    """
    if results is None or len(results) == 0:
        return pd.DataFrame()
    
    df = pd.DataFrame(results, columns=column_names)
    return df


In [4]:
def search_by_radec(ra, dec, radius, table='imastats'):
    conn = connect_to_db()
    if conn is None:
        return None
    
    try:
        cur = conn.cursor()
        
        # Validar la tabla seleccionada
        if table not in ['imastats', 'imaphot']:
            raise ValueError("La tabla debe ser 'imastats' o 'imaphot'.")

        query = sql.SQL("""
            SELECT * FROM {}
            WHERE q3c_radial_query(ra, dec, {}, {}, {})
        """).format(sql.Identifier(table), sql.Literal(ra), sql.Literal(dec), sql.Literal(radius))
        
        cur.execute(query)
        results = cur.fetchall()
        
        # Obtener los nombres de las columnas
        column_names = [desc[0] for desc in cur.description]
        
        # Convertir resultados a DataFrame
        df = results_to_dataframe(results, column_names)
        
        return df
    except (Exception, psycopg2.Error) as error:
        print("Error ejecutando la consulta:", error)
        return pd.DataFrame()
    finally:
        if conn:
            cur.close()
            conn.close()


In [5]:
def search_by_date_range(start_date, end_date):
    conn = connect_to_db()
    if conn is None:
        return None
    
    try:
        cur = conn.cursor()
        query = sql.SQL("""
            SELECT i.*, p.flux, p.dflux
            FROM imastats i
            JOIN imaphot p ON i.id = p.id
            WHERE i.date_obs BETWEEN {} AND {}
            LIMIT 100
        """).format(sql.Literal(start_date), sql.Literal(end_date))
        
        cur.execute(query)
        results = cur.fetchall()
        
        # Obtener los nombres de las columnas
        column_names = [desc[0] for desc in cur.description]
        
        # Convertir resultados a DataFrame
        df = results_to_dataframe(results, column_names)
        
        return df
    except (Exception, psycopg2.Error) as error:
        print("Error ejecutando la consulta:", error)
        return pd.DataFrame()
    finally:
        if conn:
            cur.close()
            conn.close()


In [6]:
def search_by_filename(filename_part):
    conn = connect_to_db()
    if conn is None:
        return None
    
    try:
        cur = conn.cursor()
        query = sql.SQL("""
            SELECT * FROM imastats
            WHERE file_path LIKE {}
            LIMIT 10
        """).format(sql.Literal(f'%{filename_part}%'))
        
        cur.execute(query)
        results = cur.fetchall()
        
        # Obtener los nombres de las columnas
        column_names = [desc[0] for desc in cur.description]
        
        # Convertir resultados a DataFrame
        df = results_to_dataframe(results, column_names)
        
        return df
    except (Exception, psycopg2.Error) as error:
        print("Error ejecutando la consulta:", error)
    finally:
        if conn:
            cur.close()
            conn.close()


In [7]:
def search_transients(date_after):
    conn = connect_to_db()
    if conn is None:
        return None
    
    try:
        cur = conn.cursor()
        query = sql.SQL("""
            SELECT p.*, i.file_path, i.date_obs, i.filter
            FROM imaphot p
            JOIN imastats i ON p.id = i.id
            WHERE p.trans = TRUE
              AND i.date_obs > {}
            LIMIT 50
        """).format(sql.Literal(date_after))
        
        cur.execute(query)
        results = cur.fetchall()
        
        # Obtener los nombres de las columnas
        column_names = [desc[0] for desc in cur.description]
        
        # Convertir resultados a DataFrame
        df = results_to_dataframe(results, column_names)
        
        return df
    except (Exception, psycopg2.Error) as error:
        print("Error ejecutando la consulta:", error)
    finally:
        if conn:
            cur.close()
            conn.close()


In [8]:
def search_transient_images(date_after):
    conn = connect_to_db()
    if conn is None:
        return None
    
    try:
        cur = conn.cursor()
        query = sql.SQL("""
            SELECT p.id, p.flux, p.dflux, i.file_path, i.date_obs, i.filter
            FROM imaphot p
            JOIN imastats i ON p.id = i.id
            WHERE p.trans = TRUE
              AND i.date_obs > {}
            LIMIT 50
        """).format(sql.Literal(date_after))
        
        cur.execute(query)
        results = cur.fetchall()
        
        # Obtener los nombres de las columnas
        column_names = [desc[0] for desc in cur.description]
        
        # Convertir resultados a DataFrame
        df = results_to_dataframe(results, column_names)
        
        return df
    except (Exception, psycopg2.Error) as error:
        print("Error ejecutando la consulta:", error)
        return pd.DataFrame()  # Devolver un DataFrame vacío en caso de error
    finally:
        if conn:
            cur.close()
            conn.close()


In [9]:
def search_by_filenames(filenames):
    conn = connect_to_db()
    if conn is None:
        return None
    
    try:
        cur = conn.cursor()
        
        # Crear una cadena de texto con los nombres de archivo para la consulta
        # Usamos sql.SQL y sql.Identifier para evitar inyecciones SQL
        formatted_filenames = ', '.join(sql.Literal(f).as_string(conn) for f in filenames)
        
        query = sql.SQL("""
            SELECT p.*, i.*
            FROM imaphot p
            JOIN imastats i ON p.id = i.id
            WHERE i.file_path IN ({})
        """).format(sql.SQL(formatted_filenames))
        
        cur.execute(query)
        results = cur.fetchall()
        
        # Obtener los nombres de las columnas
        column_names = [desc[0] for desc in cur.description]
        
        # Convertir resultados a DataFrame
        df = results_to_dataframe(results, column_names)
        
        return df
    except (Exception, psycopg2.Error) as error:
        print("Error ejecutando la consulta:", error)
        return pd.DataFrame()  # Devolver un DataFrame vacío en caso de error
    finally:
        if conn:
            cur.close()
            conn.close()


In [27]:
ra, dec = 14.123, 31.345
radius = 1.0
df_imaphot = search_by_radec(ra, dec, radius, table='imaphot')
df_imaphot.head()

,id,ra,dec,flux,dflux,trans
0,9c4f892082796668885f9b067d0c9411610ccff5,15.002814,31.964289,6452.02440,104.647835,True
1,9c4f892082796668885f9b067d0c9411610ccff5,15.023208,31.961424,257.24380,48.287327,True
2,9c4f892082796668885f9b067d0c9411610ccff5,15.013663,31.959118,322.25772,49.427692,True
3,9c4f892082796668885f9b067d0c9411610ccff5,15.028482,31.958752,298.69363,49.155140,True
4,9c4f892082796668885f9b067d0c9411610ccff5,15.009834,31.958187,223.33064,48.075890,True


In [29]:
ra, dec = 14.123, 31.345
radius = 1.0
df_imastats = search_by_radec(ra, dec, radius, table='imastats')
df_imastats.head()

,id,file_path,naxis1,naxis2,telescop,instrume,camera,filter,date_obs,exptime,object,ra,dec,fwhm,maglim,header
0,c4933ebe7052d619293af33244158023de0f5a27,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
1,4866b9afa4df8bb01ea599038c3dfbcc31d873c9,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
2,d27d835b4e75f52ded93510ef8bd03326eb5d4dc,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
3,f401dd310d8d990d12dbc22932e0e04284a06ce0,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
4,fcc7bb46fda01ce017cfe6fe5305410be08d66d7,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."


In [24]:
start_date = '2024-01-01'
end_date = '2024-12-31'
date_results = search_by_date_range(start_date, end_date)
date_results.head()

,id,file_path,naxis1,naxis2,telescop,instrume,camera,filter,date_obs,exptime,object,ra,dec,fwhm,maglim,header,flux,dflux
0,9c4f892082796668885f9b067d0c9411610ccff5,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9...",189.67017,47.978060
1,9c4f892082796668885f9b067d0c9411610ccff5,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9...",304.80630,49.653280
2,9c4f892082796668885f9b067d0c9411610ccff5,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9...",218.46255,48.816345
3,9c4f892082796668885f9b067d0c9411610ccff5,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9...",268.01172,48.764595
4,9c4f892082796668885f9b067d0c9411610ccff5,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9...",200.73787,48.126785


In [26]:
if len(date_results.index) >0:
    print(date_results['file_path'].unique())

['/data/images/gpuphot_processed/prered/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu_photometrized_photometrized_photometrized_photometrized_photometrized_photometrized.fits']


In [30]:
filename_part = 'Mrk352'
file_results = search_by_filename(filename_part)
file_results.head()

,id,file_path,naxis1,naxis2,telescop,instrume,camera,filter,date_obs,exptime,object,ra,dec,fwhm,maglim,header
0,c4933ebe7052d619293af33244158023de0f5a27,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
1,4866b9afa4df8bb01ea599038c3dfbcc31d873c9,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
2,d27d835b4e75f52ded93510ef8bd03326eb5d4dc,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
3,f401dd310d8d990d12dbc22932e0e04284a06ce0,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."
4,fcc7bb46fda01ce017cfe6fe5305410be08d66d7,/data/images/gpuphot_processed/prered/TTT1_iKo...,2048,2048,TTT1,iKon936-TTT1,iKon936-1,SDSSu,2024-12-08 21:49:19.543993,60.0,Mrk352,14.982686,31.831316,3.992192,19.76,"""AZ""=>""290.643807"", ""DC""=>""0.006"", ""RA""=>""14.9..."


In [37]:
if len(file_results.index) >0:
    print(file_results['file_path'].unique())

['/data/images/gpuphot_processed/prered/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu_photometrized_photometrized_photometrized_photometrized_photometrized_photometrized_photometrized.fits'
 '/data/images/gpuphot_processed/prered/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu_photometrized_photometrized_photometrized_photometrized.fits'
 '/data/images/gpuphot_processed/prered/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu.fits'
 '/data/images/gpuphot_processed/prered/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu_photometrized_photometrized_photometrized_photometrized_photometrized.fits'
 '/data/images/gpuphot_processed/prered/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu_ph

In [31]:
date_after = '2024-01-01'
transient_results = search_transients(date_after)
transient_results.head()

,id,ra,dec,flux,dflux,trans,file_path,date_obs,filter
0,9c4f892082796668885f9b067d0c9411610ccff5,15.124927,31.971039,189.67017,47.978060,True,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
1,9c4f892082796668885f9b067d0c9411610ccff5,15.125499,31.969734,304.80630,49.653280,True,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
2,9c4f892082796668885f9b067d0c9411610ccff5,15.132832,31.968334,218.46255,48.816345,True,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
3,9c4f892082796668885f9b067d0c9411610ccff5,15.113099,31.967617,268.01172,48.764595,True,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
4,9c4f892082796668885f9b067d0c9411610ccff5,15.124370,31.967434,200.73787,48.126785,True,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu


In [32]:
date_after = '2024-01-01'
df_transient_images = search_transient_images(date_after)
df_transient_images.head()

,id,flux,dflux,file_path,date_obs,filter
0,9c4f892082796668885f9b067d0c9411610ccff5,189.67017,47.978060,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
1,9c4f892082796668885f9b067d0c9411610ccff5,304.80630,49.653280,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
2,9c4f892082796668885f9b067d0c9411610ccff5,218.46255,48.816345,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
3,9c4f892082796668885f9b067d0c9411610ccff5,268.01172,48.764595,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu
4,9c4f892082796668885f9b067d0c9411610ccff5,200.73787,48.126785,/data/images/gpuphot_processed/prered/TTT1_iKo...,2024-12-08 21:49:19.543993,SDSSu


In [35]:
if len(df_transient_images.index) >0:
    print(df_transient_images['file_path'].unique())

['/data/images/gpuphot_processed/prered/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu/TTT1_iKon936-1_2024-12-08-21-49-19-543993_Mrk352_SDSSu_photometrized_photometrized_photometrized_photometrized_photometrized_photometrized.fits']


In [36]:
filenames = ['imagen1.fits', 'imagen2.fits', 'imagen3.fits']
df_images_data = search_by_filenames(filenames)

df_images_data.head()

""
